In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import h

K = 2        # genes per panel

In [3]:
TEST_SIZE = 0.2      # 80% train, 20% test

adata_mel = h.prepare(h.load_melanoma())
train_mel, test_mel = h.split(adata_mel, test_size=TEST_SIZE)
print(adata_mel.n_obs, "cells |", len(train_mel), "train |", len(test_mel), "test")

4513 cells | 3610 train | 903 test


### Candidate genes: four ways
- markers_hvg: highly variable genes + hand-picked markers (my first way)
- wilcoxon: Wilcoxon test on all genes, no hand-picking
- lasso: machine learning (L1 logistic regression) picks the genes
- stable: Wilcoxon repeated on many random halves of the training cells; keep the most frequent winners
All use the training cells only.

In [4]:
POOLS = ["markers_hvg", "wilcoxon", "lasso", "stable"]

cands_mel = {}
for pool in POOLS:
    cands_mel[pool] = h.candidates(adata_mel, train_mel, method=pool, n_boot=50)
    print(f"{pool:12s}", cands_mel[pool])

markers_hvg  [np.str_('SPARC'), np.str_('APOD'), np.str_('CALD1'), np.str_('PMEL'), np.str_('TIMP1'), np.str_('MLANA'), np.str_('CTSL1'), np.str_('CCND1'), np.str_('PTPRC'), np.str_('CD74'), np.str_('CD3E'), np.str_('LYZ'), np.str_('UGDH-AS1'), np.str_('CABP4'), np.str_('SHISA9'), np.str_('CCL22')]
wilcoxon     ['S100B', 'SERPINE2', 'CD63', 'PLP1', 'VAT1', 'LGALS3BP', 'GPNMB', 'PRAME', 'TMSB4X', 'PTPRC', 'ARHGDIB', 'LAPTM5', 'IKZF3', 'STK17B', 'CXCR4', 'B2M']
lasso        ['PLP1', 'PRAME', 'SERPINE2', 'SOX10', 'CDH19', 'S100B', 'EDNRB', 'MAGEA3', 'TMSB4X', 'B2M', 'CD74', 'FBLIM1', 'HLA-C', 'CXCL12', 'COL1A1', 'CREB3L1']
stable       ['S100B', 'SERPINE2', 'CD63', 'VAT1', 'PLP1', 'LGALS3BP', 'GPNMB', 'PRAME', 'TMSB4X', 'PTPRC', 'ARHGDIB', 'LAPTM5', 'IKZF3', 'STK17B', 'CXCR4', 'B2M']


In [5]:
qubo_mel = {}
for pool, cands in cands_mel.items():
    alpha, cv = h.choose_alpha_cv(adata_mel, train_mel, cands, K)
    Q, const, info = h.build_qubo(adata_mel, train_mel, cands, K, alpha, "auto")
    panel = h.best_panel(Q, const, cands, K)
    qubo_mel[pool] = {"Q": Q, "const": const, "info": info, "alpha": alpha, "panel": panel}
    print(f"{pool:12s} alpha={alpha:<5} lambda={info['lam']:.3f}  QUBO pick: {panel}")

markers_hvg  alpha=0.05  lambda=0.854  QUBO pick: [np.str_('APOD'), np.str_('PTPRC')]
wilcoxon     alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']
lasso        alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']
stable       alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']


In [6]:
lasso_mel = h.lasso_panel(adata_mel, train_mel, K)    # ML rival, picks from all genes

rows = []
for pool, q in qubo_mel.items():
    cands, r = cands_mel[pool], q["info"]["r"]
    panels = {"classical: top 2": [cands[i] for i in np.argsort(-r)[:K]],
              "QUBO (exact)": q["panel"]}
    for method, p in panels.items():
        m, s = h.repeated_splits(adata_mel, p, test_size=TEST_SIZE)
        rows.append({"pool": pool, "method": method, "panel": ", ".join(map(str, p)),
                     "test": round(h.evaluate(adata_mel, p, train_mel, test_mel), 3),
                     "10 splits": f"{m:.3f} ± {s:.3f}"})

m, s = h.repeated_splits(adata_mel, lasso_mel, test_size=TEST_SIZE)
rows.append({"pool": "all genes", "method": "classical: lasso", "panel": ", ".join(lasso_mel),
             "test": round(h.evaluate(adata_mel, lasso_mel, train_mel, test_mel), 3),
             "10 splits": f"{m:.3f} ± {s:.3f}"})

acc_mel = pd.DataFrame(rows)
acc_mel

,pool,method,panel,test,10 splits
0,markers_hvg,classical: top 2,"APOD, SPARC",0.895,0.906 ± 0.011
1,markers_hvg,QUBO (exact),"APOD, PTPRC",0.921,0.931 ± 0.008
2,wilcoxon,classical: top 2,"TMSB4X, S100B",0.968,0.976 ± 0.007
3,wilcoxon,QUBO (exact),"S100B, TMSB4X",0.968,0.976 ± 0.007
4,lasso,classical: top 2,"TMSB4X, S100B",0.968,0.976 ± 0.007
5,lasso,QUBO (exact),"S100B, TMSB4X",0.968,0.976 ± 0.007
6,stable,classical: top 2,"TMSB4X, S100B",0.968,0.976 ± 0.007
7,stable,QUBO (exact),"S100B, TMSB4X",0.968,0.976 ± 0.007
8,all genes,classical: lasso,"TMSB4X, S100B",0.968,0.976 ± 0.007


### QAOA (stable pool)
Same QUBO, now with the smallest safe lambda (about 0.96 instead of 2).
Standard X mixer: p = 1, 2, 3. XY mixer (always keeps exactly 2 genes): p = 1, 2.
Each p starts from the previous best angles (INTERP) plus random restarts.

In [7]:
q = qubo_mel["stable"]
E_mel = h.all_energies(q["Q"], q["const"])

base = h.baseline_row(E_mel, k=K)
tab_x, res_x = h.qaoa_ladder(E_mel, p_max=3, k=K)
tab_xy, res_xy = h.qaoa_ladder(E_mel, p_max=2, mixer="xy", init="dicke", k=K)

print("random guess:", {k_: round(v, 5) for k_, v in base.items()})
print("\nX mixer");  display(tab_x.round(5))
print("XY mixer");   display(tab_xy.round(5))


random guess: {'avg energy': 32.07696, 'approx ratio': np.float64(0.80856), 'P(best panel)': 2e-05, 'P(k genes)': 0.00183, 'shots for 99%': 301803}

X mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-0.43120,0.99229,0.00297,0.34681,1550,7.14894
2,-1.01938,0.99562,0.00368,0.42180,1250,54.80256
3,-1.20845,0.99668,0.00406,0.44704,1132,80.40433


XY mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-1.56018,0.99867,0.01074,1.0,427,16.85909
2,-1.57459,0.99875,0.01416,1.0,323,30.38323
